# ALGOTUNES
### Olivia Huang and Caitlynn Year

Finding what songs and artists you should listen to based on your artist input through looking for songs and artists of similar features

# Exploratory Data Analysis

In [3]:
import pandas as pd

df = pd.read_csv("SpotifyFeatures.csv")

In [4]:
print(df.head())

   genre        artist_name                        track_name  \
0  Movie     Henri Salvador       C'est beau de faire un Show   
1  Movie  Martin & les fées  Perdu d'avance (par Gad Elmaleh)   
2  Movie    Joseph Williams    Don't Let Me Be Lonely Tonight   
3  Movie     Henri Salvador    Dis-moi Monsieur Gordon Cooper   
4  Movie       Fabien Nataf                         Ouverture   

                 track_id  popularity  acousticness  danceability  \
0  0BRjO6ga9RKCKjfDqeFgWV           0         0.611         0.389   
1  0BjC1NfoEOOusryehmNudP           1         0.246         0.590   
2  0CoSDzoNIKCRs124s9uTVy           3         0.952         0.663   
3  0Gc6TVm52BwZD07Ki6tIvf           0         0.703         0.240   
4  0IuslXpMROHdEPvSl1fTQK           4         0.950         0.331   

   duration_ms  energy  instrumentalness key  liveness  loudness   mode  \
0        99373   0.910             0.000  C#    0.3460    -1.828  Major   
1       137373   0.737             0.000  F#

In [5]:
print(df["popularity"].describe())

count    232725.000000
mean         41.127502
std          18.189948
min           0.000000
25%          29.000000
50%          43.000000
75%          55.000000
max         100.000000
Name: popularity, dtype: float64


In [6]:
print(df.sort_values(by='popularity', ascending=False))

        genre      artist_name                                track_name  \
9027    Dance    Ariana Grande                                   7 rings   
107804    Pop    Ariana Grande                                   7 rings   
86951     Rap      Post Malone                                      Wow.   
107803    Pop      Post Malone                                      Wow.   
107802    Pop    Ariana Grande  break up with your girlfriend, i'm bored   
...       ...              ...                                       ...   
195435  Movie    Sally Dworsky                                Inside Out   
195434  Movie     Mike Douglas                            September Song   
195433  Movie      Keith David                       The Christmas Story   
195432  Movie  Charlton Heston               Chorus: Come And Go With Me   
0       Movie   Henri Salvador               C'est beau de faire un Show   

                      track_id  popularity  acousticness  danceability  \
9027    14msK

In [7]:
print(df.sort_values(by='danceability', ascending=False))

                   genre          artist_name  \
75396   Children's Music          Juice Music   
75762   Children's Music          Juice Music   
26911         Electronic              Quantic   
178675              Jazz              Quantic   
90205            Hip-Hop              Pitbull   
...                  ...                  ...   
220779             World     Stars Of The Lid   
205858        Soundtrack  James Newton Howard   
200052        Soundtrack        Ramin Djawadi   
221323             World              Eluvium   
217809             World     Stars Of The Lid   

                                  track_name                track_id  \
75396                            Fuzzy Wuzzy  2QyXjGX0rcQq7GVCpmLyyQ   
75762                   I've Been Everywhere  1RTrI6ixzz2OFdZunqwR5M   
26911                               Sol Clap  5a4BSdNOUDHzGwEWCJ6ym5   
178675                              Sol Clap  5a4BSdNOUDHzGwEWCJ6ym5   
90205                                Go Girl  1MgM0

# Cleaning the dataset
Removing duplicates by only keeping 1 copy of each song, as well as keeping the song with the artist's most common genre

In [9]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("SpotifyFeatures.csv")

In [10]:
NUM_FEATURES = [
    "acousticness","danceability","duration_ms","energy","instrumentalness",
    "liveness","loudness","speechiness","tempo","valence"
]
CAT_FEATURES = ["genre", "key", "mode", "time_signature"]
MIN_TRACKS_PER_ARTIST = 20

for c in NUM_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if "popularity" in df.columns:
    df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce")
else:
    df["popularity"] = np.nan

df = df.dropna(subset=["artist_name","track_name"] + NUM_FEATURES + CAT_FEATURES).copy()

df = df.drop_duplicates().copy()

if "track_id" in df.columns:
    df["non_nulls"] = df.notna().sum(axis=1)
    df = (df.sort_values(["track_id","popularity","non_nulls"], ascending=[True, False, False])
            .drop_duplicates(subset=["track_id"], keep="first")
            .drop(columns=["non_nulls"])
            .copy())
else:
    df["non_nulls"] = df.notna().sum(axis=1)
    df = (df.sort_values(["artist_name","track_name","popularity","non_nulls"], ascending=[True, True, False, False])
            .drop_duplicates(subset=["artist_name","track_name"], keep="first")
            .drop(columns=["non_nulls"])
            .copy())

# Keep artists with enough tracks
artist_counts = df["artist_name"].value_counts()
df = df[df["artist_name"].isin(artist_counts[artist_counts >= MIN_TRACKS_PER_ARTIST].index)].copy()
df = df.reset_index(drop=True)

print("Cleaned rows:", len(df))
print("Artists:", df["artist_name"].nunique())
if "track_id" in df.columns:
    print("Duplicate track_id rows (should be 0):", df.duplicated(subset=["track_id"]).sum())


Cleaned rows: 122589
Artists: 2300
Duplicate track_id rows (should be 0): 0


# Preprocess

In [12]:

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CAT_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

X_tracks = preprocessor.fit_transform(df).tocsr()
print("Track matrix shape:", X_tracks.shape)

Track matrix shape: (122589, 56)


# Track Vector Features

In [14]:
artists = np.array(sorted(df["artist_name"].unique()))
artist_to_idx = {a: i for i, a in enumerate(artists)}
artist_codes = df["artist_name"].map(artist_to_idx).to_numpy(dtype=int)

n_tracks = X_tracks.shape[0]
n_artists = len(artists)

A = sp.csr_matrix(
    (np.ones(n_tracks, dtype=np.float32), (artist_codes, np.arange(n_tracks))),
    shape=(n_artists, n_tracks)
)

artist_sums = A @ X_tracks
artist_counts = np.bincount(artist_codes, minlength=n_artists).astype(np.float32)
artist_counts[artist_counts == 0] = 1.0

X_artist = (sp.diags(1.0 / artist_counts) @ artist_sums).tocsr()
print("Artist matrix shape:", X_artist.shape)

Artist matrix shape: (2300, 56)


# Similar artists function

In [16]:
def top_similar_artists(artist_name, k=5):
    if artist_name not in artist_to_idx:
        raise ValueError(f"Artist '{artist_name}' not found in cleaned data.")
    i = artist_to_idx[artist_name]
    sims = cosine_similarity(X_artist, X_artist[i]).ravel()
    sims[i] = -np.inf
    top_idx = np.argsort(sims)[::-1][:k]
    return pd.Series(sims[top_idx], index=artists[top_idx], name="similarity_score")

# Recommendations based on Artist

In [117]:
def recommend_songs(
    artist_name,
    neighbor_k=5,
    n_songs=10,
    popularity_weight=0.25,  # higher = more popular songs
    min_popularity=None,
    exclude_same_artist=True
):
    if artist_name not in artist_to_idx:
        raise ValueError(f"Artist '{artist_name}' not found in cleaned data.")

    # neighbor artists
    neighbors = top_similar_artists(artist_name, k=neighbor_k).index.tolist()

    # candidate songs from neighbor artists
    cand = df[df["artist_name"].isin(neighbors)].copy()
    if exclude_same_artist:
        cand = cand[cand["artist_name"] != artist_name].copy()
    if min_popularity is not None:
        cand = cand[cand["popularity"].fillna(-1) >= min_popularity].copy()

    if cand.empty:
        return pd.DataFrame(columns=["artist_name","track_name","genre","popularity","sim","final_score"]), neighbors

    target_vec = X_artist[artist_to_idx[artist_name]]  # (1 x features)
    X_cand = X_tracks[cand.index.to_numpy()]           # (n_cand x features)

    sim = cosine_similarity(X_cand, target_vec).ravel()

    pop = cand["popularity"].fillna(0).to_numpy(dtype=np.float32)
    pop01 = np.clip(pop / 100.0, 0, 1)
    alpha = float(np.clip(popularity_weight, 0, 1))

    final = (1 - alpha) * sim + alpha * pop01

    out = cand.loc[:, ["artist_name","track_name","genre","popularity"]].copy()
    out["sim"] = sim
    out["final_score"] = final

    out = out.sort_values("final_score", ascending=False).head(n_songs).reset_index(drop=True)
    return out, neighbors

# Recommendations based on Song

In [ ]:
def recommend_from_song(track_name, artist_name=None, n_songs=10, exclude_same_artist=True, min_popularity=None):
    # 1) find candidate rows that match the track name (case-insensitive)
    mask = df["track_name"].str.lower().eq(track_name.lower())
    cand = df[mask].copy()

    if artist_name is not None:
        cand = cand[cand["artist_name"].str.lower().eq(artist_name.lower())].copy()

    if cand.empty:
        raise ValueError("Song not found. Try adding the artist_name or check spelling.")

    pick_idx = cand["popularity"].fillna(-1).idxmax() # pick most popular

    q = X_tracks[pick_idx]  # (1 x features)

    sims = cosine_similarity(X_tracks, q).ravel()

    sims[pick_idx] = -np.inf

    out = df.loc[:, ["artist_name","track_name","genre","popularity"]].copy()
    out["sim"] = sims

    if exclude_same_artist:
        out = out[out["artist_name"] != df.loc[pick_idx, "artist_name"]]

    if min_popularity is not None:
        out = out[out["popularity"].fillna(-1) >= min_popularity]

    out = out.sort_values("sim", ascending=False).head(n_songs).reset_index(drop=True)
    return df.loc[pick_idx, ["artist_name","track_name","genre","popularity"]], out


# User Input

In [133]:
artist = "Justin Bieber"  # change to user artist

sims = top_similar_artists(artist, k=8)
recs, neighbors = recommend_songs(
    artist,
    neighbor_k=8,
    n_songs=12,
    popularity_weight=0.25,
    min_popularity=20
)

print("\nInput artist:", artist)
print("\nTop similar artists:")
print(sims.reset_index().rename(columns={"index":"similar_artist"}).to_string(index=False))

print("\nNeighbors used for song recs:", neighbors)

print("\nRecommended songs:")
print(recs.to_string(index=False))


Input artist: Justin Bieber

Top similar artists:
   similar_artist  similarity_score
         Dua Lipa          0.997288
       Bea Miller          0.997274
    Ariana Grande          0.996883
        Lady Gaga          0.996818
  Backstreet Boys          0.996804
Sabrina Carpenter          0.996612
       Bebe Rexha          0.996434
           Matoma          0.996318

Neighbors used for song recs: ['Dua Lipa', 'Bea Miller', 'Ariana Grande', 'Lady Gaga', 'Backstreet Boys', 'Sabrina Carpenter', 'Bebe Rexha', 'Matoma']

Recommended songs:
  artist_name                               track_name genre  popularity      sim  final_score
Ariana Grande                            thank u, next Dance          95 0.972049     0.966537
Ariana Grande break up with your girlfriend, i'm bored Dance          99 0.954561     0.963421
Ariana Grande                                bloodline Dance          91 0.979072     0.961804
Ariana Grande                                 bad idea Dance          91 

In [131]:
user_song = "One Dance"

query_song, rec_songs = recommend_from_song(user_song)
print("Query song:")
print(query_song.to_string())
print("\nRecommended songs:")
print(rec_songs.to_string(index=False))

Query song:
artist_name        Drake
track_name     One Dance
genre            Hip-Hop
popularity            83

Recommended songs:
  artist_name                 track_name   genre  popularity      sim
   Chief Keef       Can You Be My Friend Hip-Hop          49 0.996299
       DaBaby Baby Sitter (feat. Offset) Hip-Hop          62 0.995487
 Ryan Caraveo              Perfect World Hip-Hop          57 0.992748
    Key Glock              Momma Told Me Hip-Hop          55 0.991847
     Big Sean      I Don't Fuck With You Hip-Hop          76 0.990750
        Migos                   Commando Hip-Hop          55 0.990392
    Meek Mill            I Got the Juice Hip-Hop          51 0.990089
    Key Glock          All I Kno Is Trap Hip-Hop          49 0.989878
         Tyga                      Faded Hip-Hop          63 0.989020
Vince Staples                 Yeah Right Hip-Hop          58 0.988594
